# IPNYB 101

# IMPORTING LIB'S AND DEVICE

In [53]:
import os, json, math, random, glob, time, shutil
from pathlib import Path
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
import torchvision as tv
from torchvision import transforms

from sklearn.model_selection import train_test_split

In [75]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cuda


In [55]:
!nvidia-smi

Wed Sep 17 23:50:48 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.153.02             Driver Version: 570.153.02     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3050 ...    Off |   00000000:01:00.0 Off |                  N/A |
| N/A   53C    P8              9W /   35W |    3620MiB /   4096MiB |      6%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# LOAD PATH

In [56]:
TRAIN_IMG_DIR = Path("../Datasets/train/images/")
TRAIN_LBL_DIR = Path("../Datasets/train/labels/")
TEST_IMG_DIR  = Path("../Datasets/test/images/")
SUB_TEMPLATE  = Path("../Datasets/sample_submission.csv")

# REPRODUCIBILITY

In [57]:
def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(2025)

# TRAINING HYPERPAR

In [58]:
CFG = dict(
    img_short_side=512,          # keep detail; resize by short-side, preserve aspect
    pad_to=32,                   # pad to multiple of 32 for CNN efficiency
    batch_size=2,                # tune to your GPU memory (RTX 3050: 4–8 is typical at 768p)
    num_workers=2,
    epochs=25,
    lr=3e-4,
    weight_decay=1e-4,
    model_name="resnet34",       # 'resnet34' if VRAM is tight
    dropout=0.2,
    warmup_epochs=2,
    val_split=0.1,
    amp=True,                    # mixed precision
)

# HIGH FIDEL TRANSFORM

In [59]:
class LetterboxPad:
    """Pad to (H, W) that are multiples of 'pad_to', keeping content centered."""
    def __init__(self, pad_to=32, fill=0):
        self.pad_to = pad_to; self.fill = fill
    def __call__(self, img: Image.Image):
        w, h = img.size
        new_w = math.ceil(w / self.pad_to) * self.pad_to
        new_h = math.ceil(h / self.pad_to) * self.pad_to
        if new_w == w and new_h == h:
            return img
        out = Image.new(img.mode, (new_w, new_h), color=self.fill)
        out.paste(img, ((new_w - w)//2, (new_h - h)//2))
        return out

def build_transforms(train=True):
    tfms = []
    # 1) resize by short side -> preserves aspect
    tfms.append(transforms.Lambda(
        lambda im: tv.transforms.functional.resize(
            im, size=CFG["img_short_side"], max_size=None, antialias=True)))
    # 2) (optional) gentle photometric augmentation for train
    if train:
        tfms += [
            transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.02),
            transforms.RandomHorizontalFlip(p=0.5),
        ]
    # 3) pad to multiple of 32 (letterbox)
    tfms.append(LetterboxPad(CFG["pad_to"]))
    # 4) to tensor + ImageNet norm
    tfms += [
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ]
    return transforms.Compose(tfms)


In [60]:
def _pad_to(img: torch.Tensor, H: int, W: int) -> torch.Tensor:
    """
    img: [C,H0,W0] -> pad bottom/right to [C,H,W]
    """
    _, h, w = img.shape
    pad_h, pad_w = H - h, W - w
    if pad_h == 0 and pad_w == 0:
        return img
    return F.pad(img, (0, pad_w, 0, pad_h), value=0)

def collate_pad_train(batch):
    """
    batch: list of (image_tensor, target_tensor)
    return:
        imgs: [B,C,H*,W*] with per-batch max H*, W*
        ys:   [B,1]
    """
    imgs, ys = zip(*batch)
    H = max(t.shape[1] for t in imgs)
    W = max(t.shape[2] for t in imgs)
    imgs = torch.stack([_pad_to(t, H, W) for t in imgs], dim=0)
    ys = torch.stack(ys, dim=0)
    return imgs, ys

def collate_pad_test(batch):
    """
    batch: list of (image_tensor, filename_str)
    """
    imgs, names = zip(*batch)
    H = max(t.shape[1] for t in imgs)
    W = max(t.shape[2] for t in imgs)
    imgs = torch.stack([_pad_to(t, H, W) for t in imgs], dim=0)
    return imgs, list(names)

# DATASET CLASS

In [61]:
class CrowdCountDataset(Dataset):
    def __init__(self, img_paths, lbl_dir: Path | None, train=True):
        self.img_paths = img_paths
        self.lbl_dir = lbl_dir
        self.train = train
        self.tfms = build_transforms(train=train)

    def _load_count(self, img_path: Path):
        if self.lbl_dir is None:
            return None
        stem = img_path.stem
        lbl_fp = self.lbl_dir / f"{stem}.json"
        with open(lbl_fp, "r") as f:
            js = json.load(f)
        # Expect 'human_num' in your JSON
        return float(js.get("human_num", 0.0))

    def __len__(self): return len(self.img_paths)

    def __getitem__(self, i):
        img_path = self.img_paths[i]
        with Image.open(img_path) as im:
            im = im.convert("RGB")
        y = self._load_count(img_path)
        im = self.tfms(im)
        if y is None:
            return im, img_path.name  # test-time: return filename
        return im, torch.tensor([y], dtype=torch.float32)

In [62]:
train_imgs = sorted([Path(p) for p in glob.glob(str(TRAIN_IMG_DIR / "*")) if p.lower().endswith((".jpg",".jpeg",".png"))])
test_imgs = sorted([Path(p) for p in glob.glob(str(TRAIN_IMG_DIR / "*")) if p.lower().endswith((".jpg",".jpeg",".png"))])
assert len(train_imgs) >= 50, f"Found too few train images: {len(train_imgs)}"

tr_imgs, val_imgs = train_test_split(train_imgs, test_size=CFG["val_split"], random_state=42)
ds_tr = CrowdCountDataset(tr_imgs, TRAIN_LBL_DIR, train=True)
ds_va = CrowdCountDataset(val_imgs, TRAIN_LBL_DIR, train=False)
ds_te = CrowdCountDataset(test_imgs, lbl_dir=None, train=False)

dl_tr = DataLoader(
    ds_tr,
    batch_size=CFG["batch_size"],
    shuffle=True,
    num_workers=CFG["num_workers"],
    pin_memory=True,
    collate_fn=collate_pad_train,     # <-- new
)

dl_va = DataLoader(
    ds_va,
    batch_size=CFG["batch_size"],
    shuffle=False,
    num_workers=CFG["num_workers"],
    pin_memory=True,
    collate_fn=collate_pad_train,     # <-- new (same as train; no aug here anyway)
)

dl_te = DataLoader(
    ds_te,
    batch_size=CFG["batch_size"],
    shuffle=False,
    num_workers=CFG["num_workers"],
    pin_memory=True,
    collate_fn=collate_pad_test,      # <-- new
)

len(train_imgs), len(tr_imgs), len(val_imgs), len(dl_te), len(dl_va)

(1900, 1710, 190, 950, 95)

# MODELLING (REGRESSION)

In [63]:
def build_model(name="resnet50", dropout=0.2):
    if name == "resnet34":
        backbone = tv.models.resnet34(weights=tv.models.ResNet34_Weights.IMAGENET1K_V1)
        in_feats = backbone.fc.in_features
        backbone.fc = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_feats, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, 1),   # scalar count
        )
    elif name == "resnet50":
        backbone = tv.models.resnet50(weights=tv.models.ResNet50_Weights.IMAGENET1K_V2)
        in_feats = backbone.fc.in_features
        backbone.fc = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_feats, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(512, 1),
        )
    else:
        raise ValueError("Unsupported model")
    return backbone

In [64]:
# Build model
model = build_model(CFG["model_name"], CFG["dropout"]).to(device)
model = model.to(memory_format=torch.channels_last)

# ---- Freeze early layers ----
for p in list(model.parameters())[:-2]:  # freeze all but last 2 parameter groups
    p.requires_grad = False

# Define loss, metric, optimizer, scheduler
criterion = nn.SmoothL1Loss(beta=2.0)
mae = nn.L1Loss(reduction="none")

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),  # only trainable params
    lr=CFG["lr"], weight_decay=CFG["weight_decay"]
)
num_steps = CFG["epochs"] * math.ceil(len(ds_tr) / CFG["batch_size"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_steps)
scaler = torch.cuda.amp.GradScaler(enabled=CFG["amp"])


Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /home/diyrad167/.cache/torch/hub/checkpoints/resnet34-b627a593.pth
100%|██████████| 83.3M/83.3M [00:56<00:00, 1.55MB/s]
/tmp/ipykernel_32615/3503190269.py:19: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=CFG["amp"])


# TRAINING AND VALIDATE LOOPS

In [65]:
ACCUM_STEPS = 2  # 2 x micro-batch (2) = effective batch 4

def run_epoch(model, loader, train=True):
    model.train(train)
    total_loss, total_abs_err, n = 0.0, 0.0, 0
    if train: optimizer.zero_grad(set_to_none=True)
    step = 0
    for imgs, targets in loader:
        try:
            imgs = imgs.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)

            with torch.cuda.amp.autocast(enabled=CFG["amp"]), torch.set_grad_enabled(train):
                preds = model(imgs)
                loss = criterion(preds, targets) / (ACCUM_STEPS if train else 1)

            if train:
                scaler.scale(loss).backward()
                step += 1
                if step % ACCUM_STEPS == 0:
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)
                    scheduler.step()

            abs_err = mae(preds.detach(), targets).sum().item()
            total_abs_err += abs_err
            total_loss += (loss.detach().item() * (ACCUM_STEPS if train else 1)) * imgs.size(0)
            n += imgs.size(0)

            del imgs, targets, preds, loss
        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                print("[WARN] OOM on this batch — skipping.")
                del imgs, targets
                if torch.cuda.is_available(): torch.cuda.empty_cache()
                continue
            else:
                raise
    return (total_loss / max(n,1)), (total_abs_err / max(n,1))


In [66]:
best_val_mae = float("inf")
os.makedirs("checkpoints", exist_ok=True)
for epoch in range(1, CFG["epochs"]+1):
    t0 = time.time()
    tr_loss, tr_mae = run_epoch(model, dl_tr, train=True)
    va_loss, va_mae = run_epoch(model, dl_va, train=False)

    if va_mae < best_val_mae:
        best_val_mae = va_mae
        torch.save({"epoch": epoch, "model": model.state_dict(), "cfg": CFG}, "checkpoints/best.pth")

    print(f"Epoch {epoch:02d} | "
          f"train loss {tr_loss:.4f} mae {tr_mae:.3f} | "
          f"val loss {va_loss:.4f} mae {va_mae:.3f} | "
          f"lr {optimizer.param_groups[0]['lr']:.2e} | "
          f"{time.time()-t0:.1f}s")

print("Best val MAE:", best_val_mae)

/tmp/ipykernel_32615/1707217361.py:13: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=CFG["amp"]), torch.set_grad_enabled(train):


[WARN] OOM on this batch — skipping.
[WARN] OOM on this batch — skipping.
[WARN] OOM on this batch — skipping.
[WARN] OOM on this batch — skipping.
[WARN] OOM on this batch — skipping.
[WARN] OOM on this batch — skipping.
[WARN] OOM on this batch — skipping.
[WARN] OOM on this batch — skipping.
[WARN] OOM on this batch — skipping.
[WARN] OOM on this batch — skipping.
[WARN] OOM on this batch — skipping.
[WARN] OOM on this batch — skipping.
[WARN] OOM on this batch — skipping.
[WARN] OOM on this batch — skipping.
[WARN] OOM on this batch — skipping.
[WARN] OOM on this batch — skipping.
[WARN] OOM on this batch — skipping.
[WARN] OOM on this batch — skipping.
[WARN] OOM on this batch — skipping.
[WARN] OOM on this batch — skipping.
[WARN] OOM on this batch — skipping.
[WARN] OOM on this batch — skipping.
Epoch 01 | train loss 130.3450 mae 131.318 | val loss 139.4377 mae 140.402 | lr 3.00e-04 | 40.8s
[WARN] OOM on this batch — skipping.
[WARN] OOM on this batch — skipping.
[WARN] OOM on t

# INSPECT

In [67]:
with torch.inference_mode():
    imgs, ys = next(iter(dl_va))
    preds = model(imgs.to(device)).cpu().squeeze(1)
for i in range(min(5, len(imgs))):
    print(f"GT={ys[i].item():.1f} | Pred={preds[i].item():.1f}")


GT=58.0 | Pred=32.1
GT=14.0 | Pred=23.5


# INFERENCE TO TEST SET

In [70]:
def _pad_to(img, H, W):
    _, h, w = img.shape
    return F.pad(img, (0, W - w, 0, H - h), value=0)

def collate_pad_test(batch):
    # batch: list of (tensor, filename_str)
    imgs, names = zip(*batch)
    H = max(t.shape[1] for t in imgs)
    W = max(t.shape[2] for t in imgs)
    imgs = torch.stack([_pad_to(t, H, W) for t in imgs], dim=0).to(memory_format=torch.channels_last)
    return imgs, list(names)

In [76]:
# ==== Ultra-safe test-time inference for 4 GB GPUs ====

# 0) Build test dataset
test_imgs = sorted([Path(p) for p in glob.glob(str(TEST_IMG_DIR / "*"))
                    if p.lower().endswith((".jpg",".jpeg",".png"))])
ds_te = CrowdCountDataset(test_imgs, lbl_dir=None, train=False)

# 1) Use batch_size=1 to avoid huge per-batch padding
#    -> no custom collate needed at all
dl_te = DataLoader(
    ds_te,
    batch_size=1,
    shuffle=False,
    num_workers=0,         # lower memory pressure
    pin_memory=True,
)

# 2) Load checkpoint safely and prep model
ckpt = torch.load("checkpoints/best.pth", map_location=device, weights_only=False)
model.load_state_dict(ckpt["model"])
model.eval()
model = model.to(memory_format=torch.channels_last)

# 3) Inference loop with AMP + OOM fallback to CPU per image
pred_rows = []
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# prefer fp16 on CUDA for memory; if your GPU prefers bf16 you can use dtype=torch.bfloat16
amp_dtype = torch.float16 if device.type == "cuda" else torch.bfloat16

with torch.inference_mode(), torch.amp.autocast("cuda", enabled=(device.type=="cuda"), dtype=amp_dtype):
    for imgs, names in dl_te:
        try:
            # imgs: [1, C, H, W]
            imgs = imgs.to(device, non_blocking=True).to(memory_format=torch.channels_last)
            preds = model(imgs).detach().cpu().squeeze(1).numpy()
        except RuntimeError as e:
            if "out of memory" in str(e).lower() and device.type == "cuda":
                # Per-image fallback to CPU if even bs=1 OOMs (rare but possible for very large frames)
                print(f"[WARN] OOM on {names[0]} — retrying on CPU.")
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                imgs_cpu = imgs.cpu()
                with torch.inference_mode():
                    preds = model.to("cpu")(imgs_cpu).detach().squeeze(1).numpy()
                model.to(device)  # move back
            else:
                raise

        preds = np.clip(preds, 0, None)
        pred_rows.append((names[0], float(preds[0])))

len(pred_rows), pred_rows[:3]


(500, [('1.jpg', 21.328125), ('10.jpg', 37.375), ('100.jpg', 38.0625)])

# SUBMIT CSV

In [78]:
# %%
import pandas as pd

# Try to infer expected column names from sample submission.
# Commonly: ["image_id", "count"] or ["Id","Predicted"] etc.
if SUB_TEMPLATE.exists():
    sub_template = pd.read_csv(SUB_TEMPLATE)
    cols = list(sub_template.columns)
    print("Sample submission columns:", cols)
    # Guess which column holds the filename key
    key_col = [c for c in cols if "id" in c.lower() or "image" in c.lower()][0]
    pred_col = [c for c in cols if c != key_col][0]
    # Map filename -> predicted
    df_pred = pd.DataFrame(pred_rows, columns=["filename","pred"])
    # Try to merge on filename; if template stores plain names or with extension differences, normalize both.
    T = sub_template.copy()
    T["_key"] = T[key_col].astype(str).str.replace(r"^\.?/","", regex=True)
    df_pred["_key"] = df_pred["filename"].astype(str)
    out = T.merge(df_pred[["_key","pred"]], on="_key", how="left").drop(columns="_key")
    out[pred_col] = out["pred"].fillna(0).round(3)
    out = out[[key_col, pred_col]]
else:
    # Fallback if no template: assume ["image_id","count"]
    out = pd.DataFrame(pred_rows, columns=["image_id","count"])
    out["count"] = out["count"].round(3)

SAVE_PATH = "../Submission/submission_resnet34_try1.csv"
out.to_csv(SAVE_PATH, index=False)
print("Wrote:", SAVE_PATH)
out.head()


Sample submission columns: ['image_id', 'predicted_count']
Wrote: ../Submission/submission_resnet34_try1.csv


,image_id,predicted_count
0,1.jpg,21.328
1,2.jpg,36.938
2,3.jpg,38.594
3,4.jpg,51.188
4,5.jpg,34.625
